# Movie Recommendation Engine Using Machine Learning

## CODTECH Internship Task 4

This notebook demonstrates a Content-Based Filtering recommendation system. It uses TF-IDF vectorization on combined text features (genres, overview, keywords) and computes Cosine Similarity to find the most related movies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### 1. Data Loading

In [ ]:
df = pd.read_csv('data/movies.csv')
display(df.head())

### 2. Data Cleaning & Text Feature Engineering

In [ ]:
for feature in ['genre', 'overview', 'keywords']:
    df[feature] = df[feature].fillna('')

df['combined_features'] = df['genre'] + ' ' + df['overview'] + ' ' + df['keywords']
display(df[['title', 'combined_features']].head())

### 3. TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['combined_features'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

### 4. Cosine Similarity Matrix

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Cosine Similarity Matrix Shape: {cosine_sim.shape}")

### 5. Visualizations

In [ ]:
# Genre Distribution
genres = df['genre'].str.split(' ').explode()
genre_counts = genres.value_counts()

plt.figure(figsize=(10, 6))
genre_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Genre Distribution')
plt.xlabel('Genres')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Similarity Statistics
sim_values = cosine_sim[np.triu_indices_from(cosine_sim, k=1)]
plt.figure(figsize=(8, 5))
plt.hist(sim_values, bins=20, color='lightgreen', edgecolor='black')
plt.title('Distribution of Cosine Similarity Scores')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()

### 6. Movie Search & Recommendation Function

In [ ]:
def get_recommendations(title):
    try:
        idx = df[df['title'].str.lower() == title.lower()].index[0]
    except IndexError:
        return f"Movie '{title}' not found."
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices].tolist()

### 7. Test the Recommendation Engine

In [ ]:
search_movie = "The Matrix"
print(f"Top 10 Recommendations for '{search_movie}':")
for i, movie in enumerate(get_recommendations(search_movie), 1):
    print(f"{i}. {movie}")